<center><h1> WR if only nth result counts </h1></center>

What would the world record be of only the nth best result from each competitor counted for the rankings?

In [1]:
#choose n

n = 2

### Imports

In [2]:
import pandas as pd
import numpy as np

In [3]:
Results = pd.read_csv('WCA_export_Results.tsv', sep='\t')

In [4]:
Competitions = pd.read_csv('WCA_export_Competitions.tsv', sep='\t')

In [5]:
df = Results.merge(Competitions, how='left', left_on='competitionId', right_on='id', validate = "m:1")
df = df.drop('id', 1)

In [6]:
df = df.rename(columns = {'name':'competitionName'})

In [7]:
rounds = pd.read_csv('WCA_export_RoundTypes.tsv', sep='\t', low_memory = False)

In [8]:
df = df.merge(rounds[['id','rank']], how='left', left_on='roundTypeId', right_on='id')
df = df.drop('id',1)

The final dataset looks like this:

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2870882 entries, 0 to 2870881
Data columns (total 38 columns):
 #   Column                 Dtype 
---  ------                 ----- 
 0   competitionId          object
 1   eventId                object
 2   roundTypeId            object
 3   pos                    int64 
 4   best                   int64 
 5   average                int64 
 6   personName             object
 7   personId               object
 8   personCountryId        object
 9   formatId               object
 10  value1                 int64 
 11  value2                 int64 
 12  value3                 int64 
 13  value4                 int64 
 14  value5                 int64 
 15  regionalSingleRecord   object
 16  regionalAverageRecord  object
 17  competitionName        object
 18  cityName               object
 19  countryId              object
 20  information            object
 21  year                   int64 
 22  month                  int64 
 23  day    

Other imports

In [10]:
ss = pd.read_csv('WCA_export_RanksSingle.tsv', sep='\t', low_memory = False)

In [11]:
aa = pd.read_csv('WCA_export_RanksSingle.tsv', sep='\t', low_memory = False)

In [12]:
pp = pd.read_csv('WCA_export_Persons.tsv', sep='\t', low_memory = False)

event list

In [13]:
eventi = list(set(df[df['year']>2020]['eventId'])) #take current events
eventi

['333bf',
 'minx',
 '333',
 '333mbf',
 'skewb',
 'sq1',
 '444bf',
 '555',
 '666',
 '333oh',
 'clock',
 '555bf',
 '444',
 '333fm',
 '222',
 'pyram',
 '777']

# Different approach than v1 (much faster)

## single

In [17]:
df1 = df[['personId','personName','eventId','value1','value2','value3','value4','value5']]

result = pd.DataFrame()

for e in eventi:
    subset = df1[df1['eventId'] == e]
    solve = pd.concat([
        subset[subset['value1']>0][['eventId', 'personId', 'personName', 'value1']].rename(columns = {'value1':'value'}), 
        subset[subset['value2']>0][['eventId', 'personId', 'personName', 'value2']].rename(columns = {'value2':'value'}), 
        subset[subset['value3']>0][['eventId', 'personId', 'personName', 'value3']].rename(columns = {'value3':'value'}), 
        subset[subset['value4']>0][['eventId', 'personId', 'personName', 'value4']].rename(columns = {'value4':'value'}), 
        subset[subset['value5']>0][['eventId', 'personId', 'personName', 'value5']].rename(columns = {'value5':'value'})
        ], axis = 0)
    
    solve['rank'] = solve.groupby('personId')['value'].rank(method = 'min', axis = 1) #rank solves for each person
    solve = solve[solve['rank'] == n].sort_values(by = 'value', ascending = True).head(1) #take best
        
    result = pd.concat([result, solve], axis = 0)

In [18]:
result = result.rename(columns = {'eventId':'event', 'personId':'id', 'personName':'name'}).sort_values(by = 'event').reset_index(drop = True)
result 

,event,id,name,value,rank
0,222,2014ZYCH01,Jan Zych,61,2.0
1,333,2012PARK03,Max Park,418,2.0
2,333bf,2015CHER07,Tommy Cherry,1467,2.0
3,333fm,2011TRON02,Sebastiano Tronto,18,2.0
4,333mbf,2016SIGG01,Graham Siggins,430360002,2.0
5,333oh,2012PARK03,Max Park,699,2.0
6,444,2012PARK03,Max Park,1686,2.0
7,444bf,2016CHAP04,Stanley Chapel,6251,2.0
8,555,2012PARK03,Max Park,3492,2.0
9,555bf,2016CHAP04,Stanley Chapel,14880,2.0


## Average

In [19]:
df2 = df[['personId','personName','eventId','average']]

result2 = pd.DataFrame()

for e in eventi:
    subset = df2[(df2['eventId'] == e) & (df2['average']>0)][['eventId', 'personId', 'personName', 'average']]
    
    subset['rank'] = subset.groupby('personId')['average'].rank(method = 'min', axis = 1) #rank averages for each person
    subset = subset[subset['rank'] == n].sort_values(by = 'average', ascending = True).head(1) #take best
    
    result2 = pd.concat([result2, subset], axis = 0)

In [20]:
result2 = result2.rename(columns = {'eventId':'event', 'personId':'id', 'personName':'name'}).sort_values(by = 'event').reset_index(drop = True)
result2

,event,id,name,average,rank
0,222,2018KHAN28,Zayn Khanani,103,2.0
1,333,2016KOLA02,Tymon Kolasiński,515,2.0
2,333bf,2015CHER07,Tommy Cherry,1750,2.0
3,333fm,2014SCHO02,Cale Schoon,2233,2.0
4,333oh,2012PONC02,Patrick Ponce,870,2.0
5,444,2012PARK03,Max Park,2014,2.0
6,444bf,2016CHAP04,Stanley Chapel,7255,2.0
7,555,2012PARK03,Max Park,3861,2.0
8,555bf,2013LINK01,Kaijun Lin (林恺俊),18321,2.0
9,666,2012PARK03,Max Park,7590,2.0
